# Project: FMCG Demand & Supply Chain Analytics

Objective: Prepare a "Business-Ready" dataset that supports two distinct modeling tracks:

1. Blind Promotion Trap: Requires a clean "True Baseline" (uncontaminated by promos or stockouts).

2. Supply Chain Disconnect: Requires explicit visibility into stockouts and lead time volatility.

Reference: docs/data_preprocessing_guidelines.md

In [9]:
# Install required packages 
#!pip install pandas numpy matplotlib seaborn plotly scipy statsmodels scikit-learn pandera great-expectations

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from itertools import product
import warnings
from scipy import stats
import pandera as pa
from datetime import timedelta
import json

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
%matplotlib inline
print("Environment ready")

Environment ready


## 1. Setup and Data Loading

In [11]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

df = pd.read_csv('/Users/mac/Desktop/datastorm/data/raw_sales_data.csv', parse_dates=['date'])


df = df.sort_values(by=['sku_id', 'store_id', 'date']).reset_index(drop=True)

print(f"Dataset Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Dataset Loaded: 1100000 rows, 33 columns


,date,year,month,day,weekofyear,weekday,is_weekend,is_holiday,temperature,rain_mm,store_id,country,city,channel,latitude,longitude,sku_id,sku_name,category,subcategory,brand,units_sold,list_price,discount_pct,promo_flag,gross_sales,net_sales,stock_on_hand,stock_out_flag,lead_time_days,supplier_id,purchase_cost,margin_pct
0,2021-01-01,2021,1,1,53,4,0,1,8.440,1.240,STORE0001,Germany,Berlin,Hypermarket,52.526,13.391,SKU0001,BrandA Soda,Beverages,Soda,BrandA,81,6.240,0.000,0,505.440,505.440,290,0,7,S050,3.310,0.469
1,2021-01-02,2021,1,2,53,5,1,0,12.610,1.120,STORE0001,Germany,Berlin,Hypermarket,52.526,13.391,SKU0001,BrandA Soda,Beverages,Soda,BrandA,134,6.240,0.200,1,836.160,668.930,253,0,7,S006,3.460,0.245
2,2021-01-03,2021,1,3,53,6,1,0,12.020,2.690,STORE0001,Germany,Berlin,Hypermarket,52.526,13.391,SKU0001,BrandA Soda,Beverages,Soda,BrandA,257,6.240,0.150,1,1603.680,1363.130,212,0,10,S044,3.700,0.257
3,2021-01-04,2021,1,4,1,0,0,0,7.760,4.650,STORE0001,Germany,Berlin,Hypermarket,52.526,13.391,SKU0001,BrandA Soda,Beverages,Soda,BrandA,284,6.240,0.150,1,1772.160,1506.340,306,0,4,S044,4.360,0.151
4,2021-01-05,2021,1,5,1,1,0,0,11.160,1.770,STORE0001,Germany,Berlin,Hypermarket,52.526,13.391,SKU0001,BrandA Soda,Beverages,Soda,BrandA,92,6.240,0.000,0,574.080,574.080,242,0,7,S047,3.870,0.380


### Data Cleaning
We will perform specific cleaning operations to ensure data integrity:
1. **Remove Duplicates**: To prevent skewed inventory counts.

2. **Fix Structural Errors**: To standardize categorical naming (e.g., "Germany" vs "germany").

In [12]:
#1. Remove Duplicates
initial_rows = df.shape[0]
df = df.drop_duplicates()
removed_rows = initial_rows - df.shape[0]
print(f"Duplicates removed: {removed_rows}")

Duplicates removed: 0


In [13]:
# 2. Fix Structural Errors (Standardization)
print("Standardizing categorical text...")
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    # Strip whitespace
    df[col] = df[col].str.strip()
    
    # Title Case (Skip IDs to preserve exact codes)
    if col not in ['sku_id', 'store_id', 'supplier_id', 'date']:
        df[col] = df[col].str.title()

print("Text standardization complete.")

Standardizing categorical text...
Text standardization complete.


## 2. Business Logic: Effective Price Calculation

Context: Raw list_price does not reflect the consumer's reality. To accurately model Price Elasticity later, we must calculate what the customer actually paid.

- Rule: effective_price = list_price * (1 - discount_pct)

In [14]:
# Check for required columns
required_cols = ['list_price', 'discount_pct']
if not all(col in df.columns for col in required_cols):
    raise ValueError(f"Missing columns for price calculation. Need: {required_cols}")

# Execute Business Logic
df['effective_price'] = df['list_price'] * (1 - df['discount_pct'])

# Validation: Effective price should never be negative
assert (df['effective_price'] >= 0).all(), "Critical Error: Negative prices detected."

print("Feature Created: 'effective_price'")

Feature Created: 'effective_price'


## 3. The "True Baseline" Identification
Context: This is the most critical step for Problem Statement 01. We cannot calculate "Incremental Lift" if we don't know what "Normal Sales" look like. A row counts as "True Baseline" ONLY if:

- No promotion was active (promo_flag == 0).

- No stockout occurred (stock_out_flag == 0).

Note: If we include stockout days in the baseline, we underestimate demand. If we include promo days, we overestimate it.

In [15]:
# Ensure flags are integers/binary
df['promo_flag'] = df['promo_flag'].astype(int)
df['stock_out_flag'] = df['stock_out_flag'].astype(int)

# Create the Baseline Mask
# 1 = Use this row for training baseline models
# 0 = Exclude this row (it is either a promo event or a supply failure)
df['is_true_baseline'] = (
    (df['promo_flag'] == 0) & 
    (df['stock_out_flag'] == 0)
).astype(int)

# Quick Stats
baseline_ratio = df['is_true_baseline'].mean()
print(f"Baseline Mask Created.")
print(f"Percentage of clean baseline data available: {baseline_ratio:.1%}")

if baseline_ratio < 0.2:
    print("WARNING: Less than 20% of data represents clean baseline. Modeling may be difficult.")

Baseline Mask Created.
Percentage of clean baseline data available: 89.2%


## 4. Supply Chain Hygiene (Problem Statement 02)

Context: For the Supply Chain Disconnect, we need to validate lead_time_days and stock_out_flag. We are NOT imputing demand here; we are ensuring the signals for "Risk" are clean.

Rule:

- lead_time_days must be positive.

Zero sales during a stockout != Zero Demand. We must ensure we don't accidentally treat stockouts as "low demand" periods in future steps.

In [16]:
# 1. Lead Time Cleaning
# Negative lead times are physically impossible (data errors)
invalid_lead_times = df[df['lead_time_days'] < 0]
if not invalid_lead_times.empty:
    print(f"Found {len(invalid_lead_times)} rows with negative lead times. Replacing with NaN.")
    df.loc[df['lead_time_days'] < 0, 'lead_time_days'] = np.nan

# 2. Stockout Logic Check
# If units_sold > 0, stock_out_flag should ideally be 0 (unless partial stockout).
# We interpret stock_out_flag = 1 as "Availability Issue Detected".
stockout_sales = df[(df['stock_out_flag'] == 1) & (df['units_sold'] > 0)]
print(f"Note: {len(stockout_sales)} rows have Sales > 0 despite being flagged as Stockout.")
print("   (This implies partial availability or intraday stockouts. Kept as is for risk modeling.)")

Note: 30580 rows have Sales > 0 despite being flagged as Stockout.
   (This implies partial availability or intraday stockouts. Kept as is for risk modeling.)


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100000 entries, 0 to 1099999
Data columns (total 35 columns):
 #   Column            Non-Null Count    Dtype         
---  ------            --------------    -----         
 0   date              1100000 non-null  datetime64[ns]
 1   year              1100000 non-null  int64         
 2   month             1100000 non-null  int64         
 3   day               1100000 non-null  int64         
 4   weekofyear        1100000 non-null  int64         
 5   weekday           1100000 non-null  int64         
 6   is_weekend        1100000 non-null  int64         
 7   is_holiday        1100000 non-null  int64         
 8   temperature       1100000 non-null  float64       
 9   rain_mm           1100000 non-null  float64       
 10  store_id          1100000 non-null  object        
 11  country           1100000 non-null  object        
 12  city              1100000 non-null  object        
 13  channel           1100000 non-null  object

## 5. Leakage Prevention & Final Export

Context: Before saving, we verify that the data structure supports time-series validation. The data must remain sorted by date to prevent "looking into the future" during feature engineering.

In [20]:
# Define the final list of columns to keep
# We include 'subcategory' and weather data ('temperature') as they might be useful for the baseline model.
output_columns = [
    'date', 
    'sku_id', 'sku_name', 'category', 'subcategory', 'brand',  # Product Hierarchy
    'store_id', 'city', 'country', 'channel',                  # Location Hierarchy
    'supplier_id',                                             # Supply Chain Hierarchy
    'units_sold', 
    'list_price', 'discount_pct', 'effective_price',           # Price Features
    'purchase_cost', 'margin_pct',                             # Profitability Features
    'promo_flag', 'stock_out_flag',                            # Risk Flags
    'lead_time_days', 'stock_on_hand',                         # Supply Chain Features
    'is_holiday', 'temperature', 'rain_mm',                    # External Regressors
    'is_true_baseline'                                         # Modeling Target Mask
]

# Save the cleaned file
df[output_columns].to_csv('cleaned_sales_data.csv', index=False)

print(f" Preprocessing Complete. File 'cleaned_sales_data.csv' saved.")
print(f"   Rows: {df.shape[0]}")
print(f"   Columns: {len(output_columns)}")

 Preprocessing Complete. File 'cleaned_sales_data.csv' saved.
   Rows: 1100000
   Columns: 25
